# Notebook 4: Open-Set Evaluation (AOPAD Cluster-Aware)
**Autoencoders for Open-Set Presentation Attack Detection**
#
Pipeline (neural net core is UNCHANGED):
  1. Load trained DWTAutoencoder
  2. Extract z0 latent features from bona-fide training data
  3. Fit PCA + K-Means + per-cluster Isolation Forests (AOPAD method)
  4. Evaluate on val/test sets -> APCER, BPCER, ACER, EER, AUC-ROC

## 1. Setup

In [1]:
import os
import sys
import torch
import numpy as np
import matplotlib.pyplot as plt

sys.path.insert(0, os.path.abspath(os.path.join(os.getcwd(), '..')))
from src import config as cfg
from src.dataset import load_manifest, create_dataloaders
from src.model import build_model
from src.evaluate import AOPADEvaluator
from src.utils import set_seed, plot_score_distributions, plot_roc_curves

set_seed()
cfg.print_config()

  Random seed set to 42
CONFIGURATION
  Device:            cuda
  Image Size:        128x128
  DWT Sub-band Size: 64x64
  Latent Dim:        512
  Batch Size:        32
  Epochs:            80
  Learning Rate:     0.0003
  Mixed Precision:   True
  Loss Weights:      α=0.3, β=1.5, γ=0.0
  Fusion Weight:     lambda=0.5
  Anomaly Weights:   w_iso=0.4, w_lmse=0.05, w_spec=0.4, w_maha=0.15
  IF Contamination:  0.05 (paper value)
  Train Users:       30 (IDs 1-30)
  Val Users:         7 (IDs 31-37)
  Test Users:        8 (IDs 38-45)
  Known Attacks:     [1]
  Unknown Attacks:   [2, 3, 4]


## 2. Load Model & Data

In [2]:
# Load best trained model (neural net core - unchanged)
model = build_model()
checkpoint = torch.load(
    os.path.join(cfg.CHECKPOINT_DIR, 'best_model.pth'),
    map_location=cfg.DEVICE,
)
model.load_state_dict(checkpoint['model_state_dict'])
print(f"[v] Loaded model from epoch {checkpoint['epoch']+1} "
      f"(loss: {checkpoint['loss']:.6f})")

# Load data
manifest = load_manifest()
train_loader, val_loader, test_loader = create_dataloaders(manifest)

[v] Loaded model from epoch 50 (loss: 0.001841)
[Dataset] Split='train', bona_fide_only=False, samples=11700
[Dataset] Split='val', bona_fide_only=False, samples=8190
[Dataset] Split='test', bona_fide_only=False, samples=9360


## 3. Fit AOPAD Ensemble & Run Evaluation

In [3]:
evaluator = AOPADEvaluator(
    model=model,
    device=cfg.DEVICE,
    noise_level=cfg.NOISE_LEVEL,
    k_max=15,                    # Search K from 2..15 (paper default)
    pca_variance=0.95,           # Keep 95% variance in PCA
)

results, val_scores, test_scores = evaluator.full_evaluation(
    train_loader=train_loader,   # NEW: needed to fit the ensemble
    val_loader=val_loader,
    test_loader=test_loader,
    save_dir=cfg.RESULTS_DIR,
)


AOPAD: FITTING ONE-CLASS ENSEMBLE


[Fit] Extracting train z0: 100%|██████████| 365/365 [00:48<00:00,  7.48it/s]


  Bona-fide samples for fitting: 3892

[0/3] Fitting StandardScaler on bona-fide features...
  Feature dims: 575  |  Scaler fitted on 3892 BF samples

[1/3] Applying PCA (variance=0.95)...
  PCA: 575D -> 243D

[2/3] Selecting optimal K (k_max=15)...
    K= 2  silhouette=0.1417
    K= 3  silhouette=0.0849
    K= 4  silhouette=0.0629
    K= 5  silhouette=0.0534
    K= 6  silhouette=0.0434
    K= 7  silhouette=0.0448
    K= 8  silhouette=0.0406
    K= 9  silhouette=0.0373
    K=10  silhouette=0.0311
    K=11  silhouette=0.0309
    K=12  silhouette=0.0324
    K=13  silhouette=0.0269
    K=14  silhouette=0.0274
    K=15  silhouette=0.0248
  -> Optimal K = 2  (silhouette=0.1417)

[3/3] Fitting 2 Isolation Forests (contamination=0.05)...
  Cluster 0: n=1989, contamination=0.05
  Cluster 1: n=1903, contamination=0.05

  [v] AOPAD ensemble fitted.
  Ensemble saved to c:\BS_Shivang\results

OPEN-SET EVALUATION (AOPAD)

[1/2] Evaluating on validation set (fit normalization limits)...


Computing scores: 100%|██████████| 256/256 [00:37<00:00,  6.91it/s]


  Score diagnostic:
    Bona-fide mean_score = 0.1171  (+/- 0.0743)
    Attack    mean_score = 0.4366
    Separation ratio     = +4.300 std  (GOOD: attacks > bona-fide)
  Calibrated threshold: 0.216489 (BPCER_target=10%, score_inverted=False)
[2/2] Evaluating on test set (using val-set limits & threshold)...


Computing scores: 100%|██████████| 293/293 [00:41<00:00,  7.07it/s]



TEST RESULTS  (AOPAD - calibrated threshold)

  Paper target (Table IV): ACER=0.160  BPCER=0.138  APCER=0.183  HTER=0.041  EER=0.034
  ------------------------------------------------------------------

  Overall (our model):
    ACER:     20.841%  (paper: 16.0%)
    BPCER:    6.346%  (paper: 13.8%)
    APCER:    35.337%  (paper: 18.3%)
    HTER:     20.841%  (paper:  4.1%)
    EER:      23.750%  (paper:  3.4%)
    AUC-ROC:  0.8479
    Accuracy: 67.88%

  Per Attack Type:
  Type                        APCER      AUC      EER   Known?      N
  ------------------------------------------------------------
  Print (Indoor)              0.10%  0.9999    0.29%  Yes (K)   2080
  Print (Outdoor)             0.77%  0.9981    1.18%   No (U)   2080
  Screen (CCE TV)            58.32%  0.7527   33.56%   No (U)   2080
  Screen (HP Monitor)        82.16%  0.6410   39.62%   No (U)   2080

  Bona Fide: mean_score=0.097882 +/- 0.066241 (n=1040)

  Validation Overall:
    ACER: 19.416%   HTER: 19.416% 

## 4. Score Distributions

In [4]:
colors_map = {0: '#2ecc71', 1: '#e74c3c', 2: '#e67e22', 3: '#9b59b6', 4: '#3498db'}

def plot_scores(scores_dict, title):
    labels  = scores_dict['labels']
    s_final = scores_dict['s_final']
    plt.figure(figsize=(12, 5))
    for lbl in sorted(np.unique(labels)):
        mask = labels == lbl
        name = cfg.ATTACK_NAMES[lbl]
        known = " *" if lbl in cfg.KNOWN_ATTACK_TYPES else (" o" if lbl > 0 else "")
        plt.hist(s_final[mask], bins=50, alpha=0.55,
                 label=f"{name}{known}", color=colors_map[lbl], density=True)
    plt.title(title, fontsize=14, fontweight='bold')
    plt.xlabel('Anomaly Score  (higher = more suspicious)')
    plt.ylabel('Density')
    plt.legend()
    plt.grid(alpha=0.3)
    plt.tight_layout()
    plt.savefig(os.path.join(cfg.RESULTS_DIR, f"score_dist_{title.split()[0].lower()}.png"),
                dpi=150)
    plt.show()

plot_scores(val_scores,  "Validation Set: Score Distribution")
plot_scores(test_scores, "Test Set: Score Distribution (Open-Set)")

C:\Users\Shivang\AppData\Local\Temp\ipykernel_58516\3595075408.py:21: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## 5. ROC Curves

In [5]:
from sklearn.metrics import roc_curve, auc as sk_auc

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 6))

# Overall ROC
labels_bin = (test_scores['labels'] > 0).astype(int)
fpr, tpr, _ = roc_curve(labels_bin, test_scores['s_final'])
roc_auc = sk_auc(fpr, tpr)
ax1.plot(fpr, tpr, 'b-', linewidth=2, label=f'Overall (AUC={roc_auc:.4f})')
ax1.plot([0, 1], [0, 1], 'k--', alpha=0.3)
ax1.set_title('Overall ROC', fontweight='bold')
ax1.set_xlabel('FPR')
ax1.set_ylabel('TPR')
ax1.legend()
ax1.grid(alpha=0.3)

# Per-attack ROC
bf_mask = (test_scores['labels'] == 0)
colors_list = ['#e74c3c', '#e67e22', '#9b59b6', '#3498db']
for attack_lbl, color in zip([1, 2, 3, 4], colors_list):
    mask = (test_scores['labels'] == attack_lbl)
    if mask.sum() == 0:
        continue
    combined = bf_mask | mask
    y_true   = (test_scores['labels'][combined] > 0).astype(int)
    y_score  = test_scores['s_final'][combined]
    fpr_t, tpr_t, _ = roc_curve(y_true, y_score)
    auc_t = sk_auc(fpr_t, tpr_t)
    name  = cfg.ATTACK_NAMES[attack_lbl]
    known = "*" if attack_lbl in cfg.KNOWN_ATTACK_TYPES else "o"
    ax2.plot(fpr_t, tpr_t, color=color, linewidth=2,
             label=f'{known} {name} (AUC={auc_t:.4f})')

ax2.plot([0, 1], [0, 1], 'k--', alpha=0.3)
ax2.set_title('Per-Attack ROC', fontweight='bold')
ax2.set_xlabel('FPR')
ax2.set_ylabel('TPR')
ax2.legend(fontsize=9)
ax2.grid(alpha=0.3)
plt.tight_layout()
plt.savefig(os.path.join(cfg.RESULTS_DIR, 'roc_curves.png'), dpi=150)
plt.show()

C:\Users\Shivang\AppData\Local\Temp\ipykernel_58516\576578651.py:42: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## 6. Results Summary Table

In [6]:
import pandas as pd

rows = []
for name, m in results['test_metrics']['per_attack'].items():
    rows.append({
        'Attack Type': name,
        'Known?':      '[v]' if m['known_attack'] else '[x] (Open-Set)',
        'APCER (%)':   f"{m['APCER']*100:.2f}",
        'AUC-ROC':     f"{m['AUC_ROC']:.4f}",
        'EER (%)':     f"{m['EER']*100:.2f}",
        'N Samples':   m['n_samples'],
    })

results_df = pd.DataFrame(rows)
print("\n" + "=" * 80)
print("OPEN-SET PAD - AOPAD CLUSTER-AWARE EVALUATION RESULTS")
print("=" * 80)
print(results_df.to_string(index=False))

o = results['test_metrics']['overall']
print(f"\nOverall: ACER={o['ACER']*100:.2f}%  EER={o['EER']*100:.2f}%  "
      f"AUC-ROC={o['AUC_ROC']:.4f}  Accuracy={o['accuracy']*100:.2f}%")

print(f"\nEnsemble config:")
print(f"  Optimal K (clusters) = {results['optimal_k']}")
print(f"  PCA components       = {results['pca_components']}")
print(f"  Contamination/cluster= {results['optimal_contamination']}")
print("\n[v] Evaluation complete. Proceed to Notebook 05 for detailed analysis.")


OPEN-SET PAD - AOPAD CLUSTER-AWARE EVALUATION RESULTS
        Attack Type         Known? APCER (%) AUC-ROC EER (%)  N Samples
     Print (Indoor)            [v]      0.10  0.9999    0.29       2080
    Print (Outdoor) [x] (Open-Set)      0.77  0.9981    1.18       2080
    Screen (CCE TV) [x] (Open-Set)     58.32  0.7527   33.56       2080
Screen (HP Monitor) [x] (Open-Set)     82.16  0.6410   39.62       2080

Overall: ACER=20.84%  EER=23.75%  AUC-ROC=0.8479  Accuracy=67.88%

Ensemble config:
  Optimal K (clusters) = 2
  PCA components       = 243
  Contamination/cluster= [0.05, 0.05]

[v] Evaluation complete. Proceed to Notebook 05 for detailed analysis.
